In [ ]:
# this is the feature extraction pipeline so we can get the embeddings directly (we can only do inference with this, no fine-tuning)
from transformers import pipeline

# model_name = "google-bert/bert-base-uncased"
model_name = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"

# core model
extractor = pipeline("feature-extraction", model=model_name, device='cuda')

### Move the model to the GPU

In [ ]:
import torch

# Setup the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # This should say 'cuda'

In [ ]:
from datasets import load_dataset

# there are all "positive" test_data"
dataset = load_dataset("Stevenf232/BC5CDR_MeSH2015_complete")
test_data = dataset["test"]

In [ ]:
test_data[0]

## The following feature extractor uses the CLS token as the "source of information":
SapBERT was trained using mean pooling across token embeddings. So don't use it here!

In [ ]:
from tqdm import tqdm

def extract_cls(extractor, test_data):
    ''' includes [CLS] pooling '''
    batch_size = 16

    # Create generators (saves RAM compared to creating full lists)
    mention_gen = (p["mention"] for p in test_data)
    entity_gen = (p["entity"] for p in test_data)

    mention_name_features = []
    entity_name_features = []


    print("Extracting mention features...")
    for output in tqdm(extractor(mention_gen, batch_size=batch_size, truncation=True, padding=True, return_tensors='pt'), total=len(test_data)):
        # The pipeline yields one result (one entity embedding) with shape: [1, sequence_length, hidden_size] at a time,
        # but this processes in batches on GPU
        cls_vector = output[0, 0, :].cpu() # [CLS] token is always in the beginning, CLS vector is the 768 dimension vector coressponding to that token
        mention_name_features.append(cls_vector)

    print("Extracting entity features...")
    for output in tqdm(extractor(entity_gen, batch_size=batch_size, truncation=True, padding=True, return_tensors='pt'), total=len(test_data)):
        cls_vector = output[0, 0, :].cpu()
        entity_name_features.append(cls_vector)

    return mention_name_features, entity_name_features

In [ ]:
mention_features, entity_features = extract_cls(extractor, test_data)

## Extractor for Sapbert (Uses mean pooling)
The pipeline extractor doesn't work well with SapBERT so we'll use tokeniser and model manually.

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")
model = AutoModel.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext").cuda()

In [ ]:
from tqdm import tqdm
import torch
import torch.nn.functional as F

def extract_sapbert(tokenizer, model, test_data, device='cuda'):
    batch_size = 16

    mention_name_features = []
    entity_name_features = []

    model.eval()

    def embed(texts):
        inputs = tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs, return_dict=True)
            hidden = outputs.last_hidden_state          # [B, seq, H]
            mask = inputs['attention_mask'].unsqueeze(-1)
            emb = (hidden * mask).sum(1) / mask.sum(1) # masked mean pooling
            emb = F.normalize(emb, p=2, dim=1)
        return emb.cpu()

    print("Extracting mention features...")
    for i in tqdm(range(0, len(test_data), batch_size)):
        # Get the slice of the dataset
        current_batch_data = test_data[i:i+batch_size]
        # Extract the 'mention' column as a list of strings
        batch = current_batch_data["mention"]
        mention_name_features.extend(embed(batch))

    print("Extracting entity features...")
    for i in tqdm(range(0, len(test_data), batch_size)):
        # Get the slice of the dataset
        current_batch_data = test_data[i:i+batch_size]
        # Extract the 'entity' column as a list of strings
        batch = current_batch_data["entity"]
        entity_name_features.extend(embed(batch))

    return mention_name_features, entity_name_features

In [ ]:
mention_features, entity_features = extract_sapbert(tokenizer, model, test_data)

# Model evaluation

Potential issue - this finds relevance using Cosine Similarity (will it have bias towards fine-tuning on cosineSimilarityLoss vs other loss functions?)

In [ ]:
def evaluate(mention_cls, entity_cls, test_data):
    # 1. Stack
    # Since they are already tensors, we just stack them.
    mentions_tensor = torch.stack(mention_cls).to('cuda')
    entities_tensor = torch.stack(entity_cls).to('cuda')

    # --- 2. Normalize ---
    # Standardize vector length so Dot Product = Cosine Similarity
    mentions_norm = torch.nn.functional.normalize(mentions_tensor, p=2, dim=1)
    entities_norm = torch.nn.functional.normalize(entities_tensor, p=2, dim=1)

    # --- 3. Matrix Multiplication ---
    # Compute similarity between ALL mentions and ALL entities instantly
    similarity_matrix = torch.mm(mentions_norm, entities_norm.T)

    # --- 4. Find Best Matches ---
    # Returns the index of the highest score for each row
    top_indices = torch.argmax(similarity_matrix, dim=1).cpu().numpy()

    # --- 5. Print Loop ---
    correct_count = 0
    print("\n--- Starting Evaluation ---\n")

    for i, top_idx in enumerate(top_indices):
        # the strange conversion to int from here on out is because the original idx is of type numpy.int64
        top_idx = int(top_idx)
        i = int(i)

        top_match_id = test_data[top_idx]["id"]
        correct_id = test_data[i]["id"]

        if top_match_id == correct_id:
            correct_count += 1

        mention_name = test_data[i]["mention"]
        top_match = test_data[top_idx]["entity"]
        correct_name = test_data[i]["entity"]

        print(f"mention_name: {mention_name}")
        print(f"correct entity name: {correct_name}")
        print(f"top_match: {top_match}")
        print("")

    # --- 6. Statistics ---
    accuracy = correct_count / len(test_data)
    print(f"total comparisons: {len(test_data)}")
    print(f"correct comparisons: {correct_count}")
    print(f"accuracy: {accuracy:.4f}")

    return accuracy

In [ ]:
evaluate(mention_features, entity_features, test_data)

## Evaluating Fine-tuned model


In [ ]:
from sentence_transformers import SentenceTransformer

fine_tuned_model_name = "Stevenf232/fine-tuned-SapBERT4"
model = SentenceTransformer(fine_tuned_model_name)

In [ ]:
from tqdm import tqdm
def encode(model, test_data):
  batch_size=16
  mention_encodings = []
  entity_encodings = []

  for i in tqdm(range(0, len(test_data), batch_size), desc="Extracting features"):
      # encode mentions
      batch = test_data[i:i + batch_size]["mention"]
      encodings = model.encode(batch, convert_to_tensor=True)
      mention_encodings.extend(encodings)

      # encode entities
      batch = test_data[i:i + batch_size]["entity"]
      encodings = model.encode(batch, convert_to_tensor=True)
      entity_encodings.extend(encodings)

  return mention_encodings, entity_encodings

In [ ]:
mention_encodings, entity_encodings = encode(model, test_data)

In [ ]:
evaluate(mention_encodings, entity_encodings, test_data)